# 深度学习神经网络选股策略

## 完整流程

| 步骤 | 内容 | 模型/技术 |
|------|------|----------|
| 1 | 读取本地股票数据 | pandas CSV 加载 |
| 2 | 特征工程与标准化 | Technical Factor + StandardScaler |
| 3 | 划分训练/验证/测试集 | 严格时序划分，无数据穿越 |
| 4 | 构建深度学习模型 | MLP / LSTM |
| 5 | 训练与早停机制 | Early Stopping, Dropout |
| 6 | 预测与回测 | Top-K 选股策略 |
| 7 | 可视化分析 | 收益曲线、学习曲线 |

> **核心特点**: 
> - 使用 PyTorch 深度学习框架
> - MLP（多层感知机）进行收益率回归预测
> - 可选 LSTM 进行时序建模
> - Dropout 和 Early Stopping 防止过拟合
> - 严格防止数据穿越

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pathlib
PROJECT_ROOT = pathlib.Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
    print(f"✅ PyTorch 版本: {torch.__version__}")
    print(f"   设备: {'GPU' if torch.cuda.is_available() else 'CPU'}")
except ImportError:
    HAS_TORCH = False
    raise ImportError("请安装 PyTorch: pip install torch")

try:
    import vectorbt as vbt
    HAS_VBT = True
except ImportError:
    HAS_VBT = False

from utils.factor_calculator import TechnicalFactorCalculator

# ============================================================
# 全局参数
# ============================================================
CSV_PATH         = './data/a_stock_history_price_20260223.csv'
HISTORY_YEARS    = 5        # 使用过去 N 年数据
PRED_HORIZON     = 5        # 预测未来 N 天收益率
TRAIN_RATIO      = 0.60
VAL_RATIO        = 0.20
TEST_RATIO       = 0.20
TOP_K_STOCKS     = 10       # 每日选前K只股票
RANDOM_SEED      = 42

# 神经网络参数
BATCH_SIZE       = 256
EPOCHS           = 100
LEARNING_RATE    = 0.001
HIDDEN_DIMS      = [128, 64, 32]  # MLP隐藏层维度
DROPOUT_RATE     = 0.3
PATIENCE         = 10       # 早停耐心值

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("\n✅ 环境配置完成")
print(f"   预测目标: 未来 {PRED_HORIZON} 日收益率（回归问题）")
print(f"   训练/验证/测试: {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")
print(f"   神经网络: MLP {HIDDEN_DIMS}")

## 步骤 1：数据加载与预处理

In [ ]:
def load_stock_data(path, years_back=5):
    """加载股票数据"""
    print(f"📂 加载数据: {path}")
    
    if not os.path.exists(path):
        alt_path = './data/a_stock_history_price.csv'
        if os.path.exists(alt_path):
            path = alt_path
            print(f"   使用替代路径: {path}")
        else:
            raise FileNotFoundError(f"找不到数据文件")
    
    df = pd.read_csv(path, low_memory=False)
    print(f"   原始: {df.shape[0]:,} 行 × {df.shape[1]} 列")
    
    # 标准化列名
    col_map = {'stock_code': 'code', 'ts_code': 'code', 'trade_date': 'date'}
    df = df.rename(columns=col_map)
    
    if 'code' in df.columns:
        df['code'] = df['code'].astype(str).str.zfill(6)
    
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    
    num_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if years_back:
        cutoff = df['date'].max() - pd.DateOffset(years=years_back)
        df = df[df['date'] >= cutoff]
    
    df = df.sort_values(['code', 'date']).reset_index(drop=True)
    
    print(f"   加载后: {df.shape[0]:,} 行 | {df['code'].nunique():,} 只股票")
    print(f"   日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
    return df

df_raw = load_stock_data(CSV_PATH, HISTORY_YEARS)
df_raw.head()

## 步骤 2：特征工程

In [ ]:
def create_features(df):
    """构建特征数据集"""
    print("\n🔧 构建特征...")
    
    all_features = []
    
    for code in df['code'].unique():
        stock_df = df[df['code'] == code].copy()
        stock_df = stock_df.sort_values('date')
        
        if len(stock_df) < 30:
            continue
        
        # 计算技术指标
        calc = TechnicalFactorCalculator()
        stock_df = calc.moving_average(stock_df, windows=[5, 10, 20, 60])
        stock_df = calc.rsi(stock_df, windows=[6, 14])
        stock_df = calc.macd(stock_df)
        stock_df = calc.bollinger_bands(stock_df)
        stock_df = calc.volatility_factors(stock_df)
        stock_df = calc.volume_factors(stock_df)
        stock_df = calc.momentum_factors(stock_df)
        
        # 目标变量
        stock_df[f'target_return_{PRED_HORIZON}d'] = (
            stock_df['close'].shift(-PRED_HORIZON) / stock_df['close'] - 1
        )
        
        # 日期特征
        stock_df['dayofweek'] = stock_df['date'].dt.dayofweek
        stock_df['month'] = stock_df['date'].dt.month
        
        all_features.append(stock_df)
    
    df_features = pd.concat(all_features, ignore_index=True)
    
    feature_cols = [c for c in df_features.columns if any(x in c for x in 
                    ['ma_', 'rsi_', 'macd_', 'bb_', 'volatility_', 'momentum_', 'volume_', 'dayofweek', 'month'])]
    
    df_features = df_features.dropna(subset=feature_cols + [f'target_return_{PRED_HORIZON}d'])
    
    print(f"   特征数: {len(feature_cols)}")
    print(f"   有效样本: {len(df_features):,}")
    
    return df_features, feature_cols

df_features, FEATURE_COLS = create_features(df_raw)
print(f"\n特征示例: {FEATURE_COLS[:10]}...")

## 步骤 3：时间序列数据集划分与标准化

In [ ]:
def time_series_split_and_scale(df, feature_cols, target_col):
    """
    时间序列划分 + 标准化
    注意: 标准化参数只能从训练集计算
    """
    print("\n⏰ 时间序列数据集划分与标准化...")
    
    dates = sorted(df['date'].unique())
    n_dates = len(dates)
    
    train_end = int(n_dates * TRAIN_RATIO)
    val_end = int(n_dates * (TRAIN_RATIO + VAL_RATIO))
    
    train_dates = dates[:train_end]
    val_dates = dates[train_end:val_end]
    test_dates = dates[val_end:]
    
    train_df = df[df['date'].isin(train_dates)].copy()
    val_df = df[df['date'].isin(val_dates)].copy()
    test_df = df[df['date'].isin(test_dates)].copy()
    
    # 划分特征和标签
    X_train_raw = train_df[feature_cols].values
    y_train = train_df[target_col].values
    X_val_raw = val_df[feature_cols].values
    y_val = val_df[target_col].values
    X_test_raw = test_df[feature_cols].values
    y_test = test_df[target_col].values
    
    # 标准化（仅在训练集上fit）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)
    
    print(f"   训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()} ({len(train_df):,})")
    print(f"   验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()} ({len(val_df):,})")
    print(f"   测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()} ({len(test_df):,})")
    print(f"\n✅ 标准化完成 (基于训练集)")
    
    return (X_train, y_train, X_val, y_val, X_test, y_test, 
            train_df, val_df, test_df, scaler)

target_col = f'target_return_{PRED_HORIZON}d'

X_train, y_train, X_val, y_val, X_test, y_test, train_df, val_df, test_df, scaler = \
    time_series_split_and_scale(df_features, FEATURE_COLS, target_col)

print(f"\n数据形状:")
print(f"   训练: X{X_train.shape}, y{y_train.shape}")
print(f"   验证: X{X_val.shape}, y{y_val.shape}")
print(f"   测试: X{X_test.shape}, y{y_test.shape}")

## 步骤 4：构建 PyTorch Dataset 和 DataLoader

In [ ]:
class StockDataset(Dataset):
    """股票数据集"""
    
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)  # (N, 1)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 创建 DataLoader
train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ DataLoader 创建完成")
print(f"   训练批次: {len(train_loader)}")
print(f"   验证批次: {len(val_loader)}")
print(f"   测试批次: {len(test_loader)}")

## 步骤 5：定义神经网络模型

In [ ]:
class MLPRegressor(nn.Module):
    """
    多层感知机回归模型
    """
    
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3):
        super(MLPRegressor, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


class LSTMRegressor(nn.Module):
    """
    LSTM 时序回归模型（可选）
    输入: (batch, seq_len, features)
    """
    
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(LSTMRegressor, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout_rate if num_layers > 1 else 0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        # x: (batch, seq_len, features)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # 取最后时刻的隐藏状态
        last_hidden = h_n[-1]  # (batch, hidden_dim)
        out = self.fc(last_hidden)
        return out

# 实例化模型
input_dim = len(FEATURE_COLS)

# MLP 模型
mlp_model = MLPRegressor(input_dim, HIDDEN_DIMS, DROPOUT_RATE)

print("✅ 模型创建完成")
print(f"\nMLP 模型结构:")
print(mlp_model)
print(f"\n总参数量: {sum(p.numel() for p in mlp_model.parameters()):,}")

## 步骤 6：训练模型（含早停机制）

In [ ]:
def train_model(model, train_loader, val_loader, epochs=100, patience=10, 
                learning_rate=0.001, device='cpu'):
    """
    训练模型，包含早停机制
    """
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    history = {'train_loss': [], 'val_loss': [], 'val_rmse': []}
    
    print(f"\n🚀 开始训练 (设备: {device})...")
    print("="*60)
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_losses = []
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
        
        avg_train_loss = np.mean(train_losses)
        
        # 验证阶段
        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_losses.append(loss.item())
                
                val_preds.extend(outputs.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())
        
        avg_val_loss = np.mean(val_losses)
        val_rmse = np.sqrt(mean_squared_error(val_targets, val_preds))
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_rmse'].append(val_rmse)
        
        # 学习率调整
        scheduler.step(avg_val_loss)
        
        # 早停检查
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.6f} | "
                  f"Val Loss: {avg_val_loss:.6f} | "
                  f"Val RMSE: {val_rmse:.4f}")
        
        if patience_counter >= patience:
            print(f"\n🛑 早停触发 (Epoch {epoch+1})")
            break
    
    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n使用设备: {device}")

# 训练 MLP 模型
mlp_model, history = train_model(
    mlp_model, train_loader, val_loader,
    epochs=EPOCHS,
    patience=PATIENCE,
    learning_rate=LEARNING_RATE,
    device=device
)

print("\n✅ 训练完成")

In [ ]:
# 绘制学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', alpha=0.8)
axes[0].plot(history['val_loss'], label='Val Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RMSE 曲线
axes[1].plot(history['val_rmse'], label='Val RMSE', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Validation RMSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./data/deep_learning_learning_curves.png', dpi=150)
plt.show()

print("✅ 学习曲线已保存")

## 步骤 7：模型评估与预测

In [ ]:
def evaluate_model(model, test_loader, device='cpu'):
    """评估模型"""
    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            predictions.extend(outputs.cpu().numpy().flatten())
            targets.extend(batch_y.numpy().flatten())
    
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    # 计算指标
    mse = mean_squared_error(targets, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(targets, predictions)
    r2 = r2_score(targets, predictions)
    
    return predictions, targets, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# 评估
print("\n📊 模型评估...")
pred_test, target_test, metrics = evaluate_model(mlp_model, test_loader, device)

print(f"\n测试集指标:")
print(f"   RMSE: {metrics['RMSE']:.4f}")
print(f"   MAE:  {metrics['MAE']:.4f}")
print(f"   R²:   {metrics['R2']:.4f}")

# 添加到测试集 DataFrame
test_df['pred_return'] = pred_test

# 预测 vs 真实值散点图
plt.figure(figsize=(8, 8))
plt.scatter(target_test, pred_test, alpha=0.3, s=10)
plt.plot([target_test.min(), target_test.max()], 
         [target_test.min(), target_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Return')
plt.ylabel('Predicted Return')
plt.title(f'Predicted vs Actual (R²={metrics["R2"]:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./data/deep_learning_prediction_scatter.png', dpi=150)
plt.show()

## 步骤 8：选股策略回测

In [ ]:
def backtest_strategy(test_df, top_k=10):
    """
    回测策略：每日选预测收益最高的top_k只股票
    """
    print(f"\n🚀 回测策略 (每日选Top {top_k})...")
    
    daily_returns = []
    
    for date, group in test_df.groupby('date'):
        if len(group) < top_k:
            continue
        
        # 选预测收益最高的top_k
        top_stocks = group.nlargest(top_k, 'pred_return')
        
        # 计算平均实际收益
        avg_return = top_stocks[f'target_return_{PRED_HORIZON}d'].mean()
        daily_returns.append({
            'date': date,
            'return': avg_return,
            'n_stocks': len(top_stocks)
        })
    
    returns_df = pd.DataFrame(daily_returns)
    returns_df.set_index('date', inplace=True)
    returns_df['cum_return'] = (1 + returns_df['return']).cumprod() - 1
    
    # 计算指标
    total_return = returns_df['cum_return'].iloc[-1]
    sharpe = returns_df['return'].mean() / returns_df['return'].std() * np.sqrt(252) 
    sharpe = sharpe if returns_df['return'].std() > 0 else 0
    max_dd = (returns_df['cum_return'] - returns_df['cum_return'].cummax()).min()
    win_rate = (returns_df['return'] > 0).mean()
    
    print(f"\n📊 回测结果:")
    print(f"   总收益率: {total_return*100:.2f}%")
    print(f"   夏普比率: {sharpe:.3f}")
    print(f"   最大回撤: {max_dd*100:.2f}%")
    print(f"   胜率: {win_rate*100:.1f}%")
    
    return returns_df

returns_df = backtest_strategy(test_df, TOP_K_STOCKS)

# 绘制回测曲线
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 累计收益
axes[0].plot(returns_df.index, returns_df['cum_return']*100, linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].set_title(f'Deep Learning Strategy Backtest (Top {TOP_K_STOCKS})')
axes[0].grid(True, alpha=0.3)

# 每日收益分布
axes[1].bar(returns_df.index, returns_df['return']*100, alpha=0.6)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Daily Return (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./data/deep_learning_backtest.png', dpi=150)
plt.show()

print("\n✅ 回测完成")

## 总结

In [ ]:
print("=" * 70)
print("📊 深度学习神经网络选股策略总结")
print("=" * 70)

print("\n🧠 模型架构:")
print(f"   类型: MLP (多层感知机)")
print(f"   输入维度: {len(FEATURE_COLS)}")
print(f"   隐藏层: {HIDDEN_DIMS}")
print(f"   Dropout: {DROPOUT_RATE}")

print("\n📈 训练设置:")
print(f"   批次大小: {BATCH_SIZE}")
print(f"   学习率: {LEARNING_RATE}")
print(f"   早停耐心: {PATIENCE}")

print("\n🎯 预测性能:")
print(f"   RMSE: {metrics['RMSE']:.4f}")
print(f"   R²: {metrics['R2']:.4f}")

print("\n💰 策略回测:")
print(f"   选股数量: 每日Top {TOP_K_STOCKS}")
final_return = returns_df['cum_return'].iloc[-1]
print(f"   总收益率: {final_return*100:.2f}%")

print("\n📁 输出文件:")
print("   - data/deep_learning_learning_curves.png")
print("   - data/deep_learning_prediction_scatter.png")
print("   - data/deep_learning_backtest.png")

print("\n" + "=" * 70)